In [ ]:
RUN_TARGET = "colab"  # "colab" | "local"


## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1
2. Runtime > Change runtime type > T4 GPU or A100
3. Run the pip-install cell — this installs the notebook dependencies used by the MTL workflow
4. Run the Drive-mount cell — approve access so checkpoints and results can be synced automatically
5. Run the runtime setup cell to download data, the shared utility module, and a fresh ESM-2 snapshot
6. Run the remaining cells normally
7. Outputs are copied to `Google Drive > My Drive > XAllergen2.0 > models/` and `results/`

### Running locally on macOS (M-series)
1. Set `RUN_TARGET = "local"` in Cell 1
2. The Colab setup cells are skipped automatically
3. MPS is used when available, otherwise CPU
4. Outputs are saved to the local `models/` and `results/` directories


In [ ]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "captum": "0.8.0",
        "transformers": "4.48.1",
    }
    _optional = ["statsmodels", "huggingface_hub"]

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    for name in _optional:
        try:
            _md.version(name)
        except _md.PackageNotFoundError:
            _missing_or_mismatched.append(name)

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run(
            [
                _sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                *_missing_or_mismatched,
            ],
            check=True,
        )
        print("Colab environment updated. Restart the runtime once, then rerun from the top.")
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    print(f"Google Drive mounted: {DRIVE_ROOT}")
    print(f"Models will sync to: {DRIVE_MODELS}")
    print(f"Results will sync to: {DRIVE_RESULTS}")
else:
    print("Local run: skipping Google Drive mount.")


In [ ]:
if RUN_TARGET == "colab":
    import os
    import shutil as _shutil
    import urllib.request as _urlreq
    from pathlib import Path

    from huggingface_hub import snapshot_download

    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    _DATA_DIR = RUNTIME_ROOT / "data"
    _MODEL_DIR = RUNTIME_ROOT / "models"
    _RESULTS_DIR = RUNTIME_ROOT / "results"
    _FRESH_ESM2_DIR = RUNTIME_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    for _d in [_DATA_DIR, _MODEL_DIR, _RESULTS_DIR, _FRESH_ESM2_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    _RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"

    _urlreq.urlretrieve(
        f"{_RAW}/baseline_notebook_utils.py",
        RUNTIME_ROOT / "baseline_notebook_utils.py",
    )
    print("Downloaded: baseline_notebook_utils.py")

    for _fname in [
        "positives_splitA.csv",
        "positives_splitB.csv",
        "negatives_splitA.csv",
        "negatives_splitB.csv",
    ]:
        _urlreq.urlretrieve(f"{_RAW}/data/{_fname}", _DATA_DIR / _fname)
        print(f"Downloaded: {_fname}")

    _fresh_path = snapshot_download(
        repo_id="facebook/esm2_t6_8M_UR50D",
        local_dir=_FRESH_ESM2_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_fresh_path)
    print(f"Downloaded fresh ESM-2 snapshot: {_fresh_path}")

    _baseline_checkpoint_on_drive = DRIVE_MODELS / "baseline_frozen_esm2.pt"
    if _baseline_checkpoint_on_drive.exists():
        _shutil.copy2(_baseline_checkpoint_on_drive, _MODEL_DIR / _baseline_checkpoint_on_drive.name)
        print(f"Copied baseline checkpoint from Drive: {_baseline_checkpoint_on_drive}")
    else:
        print("No baseline_frozen_esm2.pt found on Drive. Upload or copy it before training the MTL model.")

    _baseline_summary_on_drive = DRIVE_RESULTS / "probing_summary.csv"
    if _baseline_summary_on_drive.exists():
        _shutil.copy2(_baseline_summary_on_drive, _RESULTS_DIR / "probing_summary.csv")
        print(f"Copied baseline probing summary from Drive: {_baseline_summary_on_drive}")
    else:
        print("No probing_summary.csv found on Drive. Baseline-vs-MTL comparison will be skipped.")
else:
    print("Local run: skipping GitHub download and model snapshot setup.")


# 05 - Multi-Task Epitope Supervision for XAllergen2.0

This notebook fine-tunes the frozen-head portion of `FrozenESMAllergenMTLClassifier` on the curated XAllergen2.0 splits while keeping the ESM-2 backbone frozen.

Training setup:
- Protein-level allergenicity supervision on mixed curated positives and negatives from notebook 01
- Residue-level epitope supervision only on IEDB-derived positive proteins
- Initialization from the saved notebook 03 checkpoint: `models/baseline_frozen_esm2.pt`
- Held-out residue probing on `positives_splitB.csv`

Model:
- Frozen ESM-2 backbone: `esm2_t6_8M_UR50D`
- Reused baseline weights for backbone, `attention_pool`, and protein classifier head
- Fresh epitope head initialized at train time and optimized jointly with the protein head


In [ ]:
import os
import sys
from pathlib import Path

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))


In [ ]:
import json
import math
from pathlib import Path

from baseline_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    MAX_SEQ_LEN,
    RANDOM_STATE,
    THRESHOLD,
    FrozenESMAllergenMTLClassifier,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    compute_residue_probabilities,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    initialize_mtl_from_baseline_checkpoint,
    load_mtl_checkpoint,
    mean_metric_dicts,
    parse_epitope_label,
    print_runtime_context,
    seed_everything,
)

if RUN_TARGET == "colab":
    import matplotlib
    matplotlib.use("Agg")
else:
    configure_matplotlib_cache(Path.cwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [ ]:
# Hyperparameters
CLASSIFICATION_BATCH_SIZE      = 24
EPOCHS                         = 30
PATIENCE                       = 5
LEARNING_RATE                  = 1e-3
WEIGHT_DECAY                   = 1e-4
LAMBDA_CLS                     = 1.0
LAMBDA_EPI                     = 0.5
EPITOPE_HIDDEN_DIM             = 128
VAL_FRACTION                   = 0.1
USE_PROTEIN_POS_WEIGHT         = False
PROTEIN_IMBALANCE_TOLERANCE    = 0.1
N_RANDOM_DRAWS                 = 100

seed_everything(RANDOM_STATE)

if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  WARNING: no GPU detected - IG attribution will be slow.")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)

DATA_DIR    = PROJECT_ROOT / "data"
MODEL_DIR   = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

POSITIVE_TRAIN_CSV      = DATA_DIR / "positives_splitA.csv"
POSITIVE_TEST_CSV       = DATA_DIR / "positives_splitB.csv"
NEGATIVE_TRAIN_CSV      = DATA_DIR / "negatives_splitA.csv"
NEGATIVE_TEST_CSV       = DATA_DIR / "negatives_splitB.csv"
BASELINE_CHECKPOINT_PATH = MODEL_DIR / "baseline_frozen_esm2.pt"
CHECKPOINT_PATH          = MODEL_DIR / "mtl_frozen_esm2_epitope.pt"
METRICS_PATH             = RESULTS_DIR / "mtl_baseline_metrics.json"
PROBE_ROWS_PATH          = RESULTS_DIR / "mtl_probing_rows.csv"
PROBE_SUMMARY_PATH       = RESULTS_DIR / "mtl_probing_summary.csv"
COMPARE_SUMMARY_PATH     = RESULTS_DIR / "mtl_vs_baseline_summary.csv"
PROBE_PLOT_PATH          = RESULTS_DIR / "mtl_probing_means.png"
BASELINE_SUMMARY_CSV     = RESULTS_DIR / "probing_summary.csv"

tokenizer = build_tokenizer(HF_MODEL_NAME)


In [ ]:
def annotate_epitopes(frame: pd.DataFrame) -> pd.DataFrame:
    annotated = frame.copy()
    annotated["epitope_label"] = [
        parse_epitope_label(seq, start, end)
        for seq, start, end in zip(
            annotated["sequence"],
            annotated["epitope_start"],
            annotated["epitope_end"],
        )
    ]
    annotated["seq_len"] = annotated["sequence"].str.len().astype(int)
    annotated["n_epitope_residues"] = annotated["epitope_label"].map(lambda arr: int(arr.sum()))
    annotated["epitope_density"] = annotated["n_epitope_residues"] / annotated["seq_len"]
    return annotated


def filter_max_len(frame: pd.DataFrame, sequence_col: str = "sequence") -> pd.DataFrame:
    keep = frame[sequence_col].astype(str).str.len() <= MAX_SEQ_LEN
    return frame.loc[keep].reset_index(drop=True)


def require_columns(frame: pd.DataFrame, required: list[str], frame_name: str) -> None:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns in {frame_name}: {missing}. Available columns: {list(frame.columns)}")


def get_sequence_id_column(frame: pd.DataFrame, preferred: list[str], frame_name: str) -> str:
    for column in preferred:
        if column in frame.columns:
            return column
    raise ValueError(
        f"Could not find a sequence identifier column for {frame_name}. Tried {preferred}. Available columns: {list(frame.columns)}"
    )


def prepare_positive_frame(csv_path: Path, split_name: str) -> pd.DataFrame:
    frame = filter_max_len(pd.read_csv(csv_path))
    require_columns(frame, ["sequence", "epitope_start", "epitope_end"], csv_path.name)
    sequence_id_col = get_sequence_id_column(frame, ["accession", "sequence_id"], csv_path.name)
    frame = annotate_epitopes(frame)
    frame = frame.copy()
    frame["sequence_id"] = frame[sequence_id_col].astype(str)
    frame["protein_label"] = 1.0
    frame["has_epitope_supervision"] = 1
    frame["split_name"] = split_name
    frame["data_source"] = "positive"
    return frame


def prepare_negative_frame(csv_path: Path, split_name: str) -> pd.DataFrame:
    frame = filter_max_len(pd.read_csv(csv_path))
    require_columns(frame, ["sequence"], csv_path.name)
    sequence_id_col = get_sequence_id_column(frame, ["entry", "sequence_id", "accession"], csv_path.name)
    frame = frame.copy()
    frame["sequence_id"] = frame[sequence_id_col].astype(str)
    frame["seq_len"] = frame["sequence"].str.len().astype(int)
    frame["epitope_label"] = frame["seq_len"].map(lambda seq_len: np.zeros(seq_len, dtype=np.float32))
    frame["n_epitope_residues"] = 0
    frame["epitope_density"] = 0.0
    frame["protein_label"] = 0.0
    frame["has_epitope_supervision"] = 0
    frame["split_name"] = split_name
    frame["data_source"] = "negative"
    return frame


def build_mixed_frame(positive_frame: pd.DataFrame, negative_frame: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "sequence_id",
        "sequence",
        "protein_label",
        "epitope_label",
        "seq_len",
        "has_epitope_supervision",
        "n_epitope_residues",
        "epitope_density",
        "split_name",
        "data_source",
    ]
    mixed = pd.concat(
        [positive_frame[columns], negative_frame[columns]],
        ignore_index=True,
    )
    return mixed.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)


positive_train_full_df = prepare_positive_frame(POSITIVE_TRAIN_CSV, "splitA")
positive_test_df = prepare_positive_frame(POSITIVE_TEST_CSV, "splitB")
negative_train_full_df = prepare_negative_frame(NEGATIVE_TRAIN_CSV, "splitA")
negative_test_df = prepare_negative_frame(NEGATIVE_TEST_CSV, "splitB")

positive_train_df, positive_val_df = train_test_split(
    positive_train_full_df,
    test_size=VAL_FRACTION,
    random_state=RANDOM_STATE,
)
negative_train_df, negative_val_df = train_test_split(
    negative_train_full_df,
    test_size=VAL_FRACTION,
    random_state=RANDOM_STATE,
)

train_mixed_df = build_mixed_frame(positive_train_df, negative_train_df)
val_mixed_df = build_mixed_frame(positive_val_df, negative_val_df)
test_mixed_df = build_mixed_frame(positive_test_df, negative_test_df)
epitope_probe_df = positive_test_df.copy().reset_index(drop=True)

print("Mixed train/val/test:", len(train_mixed_df), len(val_mixed_df), len(test_mixed_df))
print("Positive train/val/test:", len(positive_train_df), len(positive_val_df), len(positive_test_df))
print("Negative train/val/test:", len(negative_train_df), len(negative_val_df), len(negative_test_df))
print("Positive train density mean:", round(float(positive_train_df["epitope_density"].mean()), 4))
print("Positive test density mean:", round(float(positive_test_df["epitope_density"].mean()), 4))


In [ ]:
class MixedAllergenEpitopeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        seq_len = int(row["seq_len"])
        residue_labels = np.asarray(row["epitope_label"], dtype=np.float32)
        if residue_labels.shape[0] != seq_len:
            raise ValueError(
                f"Residue label length mismatch for {row['sequence_id']}: labels={residue_labels.shape[0]}, seq_len={seq_len}"
            )
        return {
            "sequence_id": str(row["sequence_id"]),
            "sequence": row["sequence"],
            "protein_label": float(row["protein_label"]),
            "residue_labels": residue_labels,
            "seq_len": seq_len,
            "has_epitope_supervision": int(row["has_epitope_supervision"]),
            "data_source": row["data_source"],
        }


def collate_mixed_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    max_len = encodings["input_ids"].shape[1]
    residue_labels = torch.zeros(len(batch), max_len, dtype=torch.float32)
    residue_loss_mask = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for idx, item in enumerate(batch):
        seq_len = min(item["seq_len"], max_len)
        residue_labels[idx, :seq_len] = torch.tensor(item["residue_labels"][:seq_len], dtype=torch.float32)
        if item["has_epitope_supervision"]:
            residue_loss_mask[idx, :seq_len] = True

    return {
        "sequence_id": [item["sequence_id"] for item in batch],
        "sequence": sequences,
        "protein_label": torch.tensor([item["protein_label"] for item in batch], dtype=torch.float32),
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "residue_labels": residue_labels,
        "residue_loss_mask": residue_loss_mask,
        "has_epitope_supervision": torch.tensor(
            [item["has_epitope_supervision"] for item in batch],
            dtype=torch.float32,
        ),
        "data_source": [item["data_source"] for item in batch],
    }


def move_mixed_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    for key in [
        "protein_label",
        "input_ids",
        "attention_mask",
        "residue_labels",
        "residue_loss_mask",
        "has_epitope_supervision",
    ]:
        moved[key] = batch[key].to(device)
    return moved


train_loader = DataLoader(
    MixedAllergenEpitopeDataset(train_mixed_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_mixed_batch,
)
val_loader = DataLoader(
    MixedAllergenEpitopeDataset(val_mixed_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_mixed_batch,
)
test_loader = DataLoader(
    MixedAllergenEpitopeDataset(test_mixed_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_mixed_batch,
)


In [ ]:
train_positive_proteins = int(positive_train_df["protein_label"].sum())
train_negative_proteins = int((positive_train_df.shape[0] + negative_train_df.shape[0]) - train_positive_proteins)
protein_pos_weight_value = len(negative_train_df) / max(len(positive_train_df), 1)
use_protein_pos_weight = USE_PROTEIN_POS_WEIGHT and abs(protein_pos_weight_value - 1.0) > PROTEIN_IMBALANCE_TOLERANCE
protein_pos_weight = (
    torch.tensor(protein_pos_weight_value, dtype=torch.float32, device=DEVICE)
    if use_protein_pos_weight
    else None
)

total_epitope_residues = float(positive_train_df["n_epitope_residues"].sum())
total_non_epitope_residues = float((positive_train_df["seq_len"] - positive_train_df["n_epitope_residues"]).sum())
pos_weight_epi_value = total_non_epitope_residues / max(total_epitope_residues, 1.0)
residue_pos_weight = torch.tensor(pos_weight_epi_value, dtype=torch.float32, device=DEVICE)

print(f"Training protein positives: {len(positive_train_df)}")
print(f"Training protein negatives: {len(negative_train_df)}")
print(f"Protein pos_weight candidate: {protein_pos_weight_value:.3f}")
print(f"Using protein pos_weight: {use_protein_pos_weight}")
print(f"Training epitope residues (positive set only): {total_epitope_residues:.0f}")
print(f"Training non-epitope residues (positive set only): {total_non_epitope_residues:.0f}")
print(
    "pos_weight_epi = total_non_epitope_residues / total_epitope_residues = "
    f"{total_non_epitope_residues:.0f} / {total_epitope_residues:.0f} = {pos_weight_epi_value:.3f}"
)

if not BASELINE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Baseline checkpoint not found: {BASELINE_CHECKPOINT_PATH}. "
        "Run notebook 03 first or copy baseline_frozen_esm2.pt into models/."
    )

model, baseline_checkpoint = initialize_mtl_from_baseline_checkpoint(
    BASELINE_CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    epitope_hidden_dim=EPITOPE_HIDDEN_DIM,
)

assert not any(param.requires_grad for param in model.backbone.parameters())
trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Backbone hidden size: {model.backbone.config.hidden_size}")
print(f"Lambda cls: {LAMBDA_CLS}")
print(f"Lambda epi: {LAMBDA_EPI}")


In [ ]:
def compute_protein_loss(
    logits: torch.Tensor,
    protein_labels: torch.Tensor,
    pos_weight: torch.Tensor | None = None,
) -> torch.Tensor:
    kwargs = {"reduction": "mean"}
    if pos_weight is not None:
        kwargs["pos_weight"] = pos_weight
    return F.binary_cross_entropy_with_logits(logits, protein_labels, **kwargs)


def compute_masked_residue_loss(
    residue_logits: torch.Tensor,
    residue_labels: torch.Tensor,
    residue_loss_mask: torch.Tensor,
    pos_weight: torch.Tensor,
) -> tuple[torch.Tensor, int]:
    valid_mask = residue_loss_mask.bool()
    valid_count = int(valid_mask.sum().item())
    if valid_count == 0:
        return residue_logits.sum() * 0.0, 0

    valid_logits = residue_logits[valid_mask]
    valid_labels = residue_labels[valid_mask]
    loss = F.binary_cross_entropy_with_logits(
        valid_logits,
        valid_labels,
        reduction="mean",
        pos_weight=pos_weight,
    )
    return loss, valid_count


def train_one_epoch_mtl(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str,
    protein_pos_weight: torch.Tensor | None,
    residue_pos_weight: torch.Tensor,
    lambda_cls: float,
    lambda_epi: float,
) -> dict:
    model.train()
    total_cls_numerator = 0.0
    total_cls_examples = 0
    total_epi_numerator = 0.0
    total_epi_positions = 0
    total_total_loss = 0.0
    total_steps = 0

    for batch in loader:
        batch = move_mixed_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch["input_ids"], batch["attention_mask"])
        cls_loss = compute_protein_loss(outputs["logits"], batch["protein_label"], protein_pos_weight)
        epi_loss, epi_positions = compute_masked_residue_loss(
            outputs["residue_logits"],
            batch["residue_labels"],
            batch["residue_loss_mask"],
            residue_pos_weight,
        )
        total_loss = lambda_cls * cls_loss + lambda_epi * epi_loss
        total_loss.backward()
        optimizer.step()

        batch_size = int(batch["protein_label"].shape[0])
        total_cls_numerator += float(cls_loss.item()) * batch_size
        total_cls_examples += batch_size
        if epi_positions > 0:
            total_epi_numerator += float(epi_loss.item()) * epi_positions
            total_epi_positions += epi_positions
        total_total_loss += float(total_loss.item())
        total_steps += 1

    train_cls_loss = total_cls_numerator / max(total_cls_examples, 1)
    train_epi_loss = total_epi_numerator / max(total_epi_positions, 1)
    train_total_loss = total_total_loss / max(total_steps, 1)
    return {
        "train_total_loss": train_total_loss,
        "train_cls_loss": train_cls_loss,
        "train_epi_loss": train_epi_loss,
        "train_weighted_cls": lambda_cls * train_cls_loss,
        "train_weighted_epi": lambda_epi * train_epi_loss,
    }


@torch.no_grad()
def evaluate_mtl(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    protein_pos_weight: torch.Tensor | None,
    residue_pos_weight: torch.Tensor,
    lambda_cls: float,
    lambda_epi: float,
) -> dict:
    model.eval()
    total_cls_numerator = 0.0
    total_cls_examples = 0
    total_epi_numerator = 0.0
    total_epi_positions = 0
    total_total_loss = 0.0
    total_steps = 0

    for batch in loader:
        batch = move_mixed_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        cls_loss = compute_protein_loss(outputs["logits"], batch["protein_label"], protein_pos_weight)
        epi_loss, epi_positions = compute_masked_residue_loss(
            outputs["residue_logits"],
            batch["residue_labels"],
            batch["residue_loss_mask"],
            residue_pos_weight,
        )
        total_loss = lambda_cls * cls_loss + lambda_epi * epi_loss

        batch_size = int(batch["protein_label"].shape[0])
        total_cls_numerator += float(cls_loss.item()) * batch_size
        total_cls_examples += batch_size
        if epi_positions > 0:
            total_epi_numerator += float(epi_loss.item()) * epi_positions
            total_epi_positions += epi_positions
        total_total_loss += float(total_loss.item())
        total_steps += 1

    cls_loss_value = total_cls_numerator / max(total_cls_examples, 1)
    epi_loss_value = total_epi_numerator / max(total_epi_positions, 1)
    total_loss_value = total_total_loss / max(total_steps, 1)
    return {
        "total_loss": total_loss_value,
        "cls_loss": cls_loss_value,
        "epi_loss": epi_loss_value,
        "weighted_cls": lambda_cls * cls_loss_value,
        "weighted_epi": lambda_epi * epi_loss_value,
    }


@torch.no_grad()
def predict_mtl(
    model: nn.Module,
    loader: DataLoader,
    device: str,
) -> tuple[pd.DataFrame, dict]:
    model.eval()
    protein_rows = []
    residue_predictions = []
    flat_residue_labels = []
    flat_residue_scores = []

    for batch in loader:
        batch = move_mixed_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        protein_probs = torch.sigmoid(outputs["logits"]).detach().cpu()
        residue_probs = torch.sigmoid(outputs["residue_logits"]).detach().cpu()
        attention_mask = batch["attention_mask"].detach().cpu()
        residue_labels = batch["residue_labels"].detach().cpu()
        has_supervision = batch["has_epitope_supervision"].detach().cpu()
        protein_labels = batch["protein_label"].detach().cpu()

        for idx, sequence_id in enumerate(batch["sequence_id"]):
            prob = float(protein_probs[idx].item())
            protein_rows.append(
                {
                    "sequence_id": sequence_id,
                    "sequence": batch["sequence"][idx],
                    "label": int(protein_labels[idx].item()),
                    "pred_prob": prob,
                    "pred_label": int(prob >= THRESHOLD),
                    "logit": float(outputs["logits"][idx].detach().cpu().item()),
                }
            )

            if int(has_supervision[idx].item()) == 1:
                seq_len = int(attention_mask[idx].sum().item())
                labels = residue_labels[idx, :seq_len].numpy().astype(np.float32)
                scores = residue_probs[idx, :seq_len].numpy().astype(np.float32)
                residue_predictions.append(
                    {
                        "sequence_id": sequence_id,
                        "residue_labels": labels,
                        "residue_scores": scores,
                    }
                )
                flat_residue_labels.append(labels)
                flat_residue_scores.append(scores)

    payload = {
        "residue_predictions": residue_predictions,
        "residue_labels_flat": np.concatenate(flat_residue_labels) if flat_residue_labels else np.array([], dtype=np.float32),
        "residue_scores_flat": np.concatenate(flat_residue_scores) if flat_residue_scores else np.array([], dtype=np.float32),
    }
    return pd.DataFrame(protein_rows), payload


def compute_classification_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label"].to_numpy()
    y_prob = pred_df["pred_prob"].to_numpy()
    y_pred = pred_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "auprc": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else math.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


def compute_flattened_residue_metrics(labels: np.ndarray, scores: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    n_valid = int(labels.shape[0])
    n_positive = int(labels.sum())

    if n_valid == 0 or n_positive == 0 or n_positive == n_valid:
        return {
            "n_valid_residues": n_valid,
            "n_positive_residues": n_positive,
            "auroc": math.nan,
            "auprc": math.nan,
            "precision_at_k": math.nan,
        }

    k = max(n_positive, 1)
    top_k = np.argsort(scores)[-k:]
    return {
        "n_valid_residues": n_valid,
        "n_positive_residues": n_positive,
        "auroc": float(roc_auc_score(labels, scores)),
        "auprc": float(average_precision_score(labels, scores)),
        "precision_at_k": float(labels[top_k].mean()),
    }


In [ ]:
history = []
best_val_total = float("inf")
best_epoch = 0
epochs_without_improvement = 0
early_stopped = False

for epoch in tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch"):
    train_stats = train_one_epoch_mtl(
        model,
        train_loader,
        optimizer,
        DEVICE,
        protein_pos_weight=protein_pos_weight,
        residue_pos_weight=residue_pos_weight,
        lambda_cls=LAMBDA_CLS,
        lambda_epi=LAMBDA_EPI,
    )
    val_stats = evaluate_mtl(
        model,
        val_loader,
        DEVICE,
        protein_pos_weight=protein_pos_weight,
        residue_pos_weight=residue_pos_weight,
        lambda_cls=LAMBDA_CLS,
        lambda_epi=LAMBDA_EPI,
    )

    row = {
        "epoch": epoch,
        "train_total_loss": float(train_stats["train_total_loss"]),
        "train_cls_loss": float(train_stats["train_cls_loss"]),
        "train_epi_loss": float(train_stats["train_epi_loss"]),
        "train_weighted_cls": float(train_stats["train_weighted_cls"]),
        "train_weighted_epi": float(train_stats["train_weighted_epi"]),
        "val_total_loss": float(val_stats["total_loss"]),
        "val_cls_loss": float(val_stats["cls_loss"]),
        "val_epi_loss": float(val_stats["epi_loss"]),
        "val_weighted_cls": float(val_stats["weighted_cls"]),
        "val_weighted_epi": float(val_stats["weighted_epi"]),
    }
    history.append(row)

    if val_stats["total_loss"] < best_val_total:
        best_val_total = float(val_stats["total_loss"])
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "esm_model_name": ESM_MODEL_NAME,
                "baseline_checkpoint_path": str(BASELINE_CHECKPOINT_PATH),
                "architecture_hyperparameters": {
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                    "epitope_hidden_dim": EPITOPE_HIDDEN_DIM,
                },
                "training_history": history,
                "best_epoch": best_epoch,
                "lambda_cls": LAMBDA_CLS,
                "lambda_epi": LAMBDA_EPI,
                "protein_pos_weight": None if protein_pos_weight is None else float(protein_pos_weight.item()),
                "residue_pos_weight": float(residue_pos_weight.item()),
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:>3}/{EPOCHS} | "
        f"train_total={train_stats['train_total_loss']:.5f} | "
        f"train_cls={train_stats['train_cls_loss']:.5f} | "
        f"train_epi={train_stats['train_epi_loss']:.5f} | "
        f"train_lambda_cls={train_stats['train_weighted_cls']:.5f} | "
        f"train_lambda_epi={train_stats['train_weighted_epi']:.5f} | "
        f"val_total={val_stats['total_loss']:.5f} | "
        f"val_cls={val_stats['cls_loss']:.5f} | "
        f"val_epi={val_stats['epi_loss']:.5f} | "
        f"val_lambda_cls={val_stats['weighted_cls']:.5f} | "
        f"val_lambda_epi={val_stats['weighted_epi']:.5f} | "
        f"best={best_epoch}"
    )

    if epochs_without_improvement >= PATIENCE:
        early_stopped = True
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print(f"Best validation objective: {best_val_total:.5f} at epoch {best_epoch}")
print(f"Early stopping triggered: {early_stopped}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")


In [ ]:
model, checkpoint = load_mtl_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    epitope_hidden_dim=EPITOPE_HIDDEN_DIM,
)

history_df = pd.DataFrame(checkpoint["training_history"])
best_epoch = int(checkpoint.get("best_epoch", int(history_df.loc[history_df["val_total_loss"].idxmin(), "epoch"])))
early_stopped_flag = bool(globals().get("early_stopped", False))

val_stats = evaluate_mtl(
    model,
    val_loader,
    DEVICE,
    protein_pos_weight=protein_pos_weight,
    residue_pos_weight=residue_pos_weight,
    lambda_cls=LAMBDA_CLS,
    lambda_epi=LAMBDA_EPI,
)
test_stats = evaluate_mtl(
    model,
    test_loader,
    DEVICE,
    protein_pos_weight=protein_pos_weight,
    residue_pos_weight=residue_pos_weight,
    lambda_cls=LAMBDA_CLS,
    lambda_epi=LAMBDA_EPI,
)

val_predictions_df, val_residue_payload = predict_mtl(model, val_loader, DEVICE)
test_predictions_df, test_residue_payload = predict_mtl(model, test_loader, DEVICE)

val_classification_metrics = compute_classification_metrics(val_predictions_df)
val_residue_metrics = compute_flattened_residue_metrics(
    val_residue_payload["residue_labels_flat"],
    val_residue_payload["residue_scores_flat"],
)
test_metrics = compute_classification_metrics(test_predictions_df)
test_metrics["test_total_loss"] = float(test_stats["total_loss"])
test_metrics["test_cls_loss"] = float(test_stats["cls_loss"])
test_metrics["test_epi_loss"] = float(test_stats["epi_loss"])
test_metrics["test_weighted_cls"] = float(test_stats["weighted_cls"])
test_metrics["test_weighted_epi"] = float(test_stats["weighted_epi"])
test_metrics["best_epoch"] = best_epoch
test_metrics["n_test_sequences"] = int(len(test_predictions_df))

test_residue_metrics = compute_flattened_residue_metrics(
    test_residue_payload["residue_labels_flat"],
    test_residue_payload["residue_scores_flat"],
)

metrics_payload = {
    "esm_model_name": ESM_MODEL_NAME,
    "baseline_checkpoint_path": str(BASELINE_CHECKPOINT_PATH),
    "architecture_hyperparameters": {
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "epitope_hidden_dim": EPITOPE_HIDDEN_DIM,
    },
    "training": {
        "batch_size": CLASSIFICATION_BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "lambda_cls": LAMBDA_CLS,
        "lambda_epi": LAMBDA_EPI,
        "use_protein_pos_weight": use_protein_pos_weight,
        "protein_pos_weight": None if protein_pos_weight is None else float(protein_pos_weight.item()),
        "residue_pos_weight": float(residue_pos_weight.item()),
        "residue_pos_weight_formula": "total_non_epitope_residues / total_epitope_residues",
        "total_epitope_residues_train": float(total_epitope_residues),
        "total_non_epitope_residues_train": float(total_non_epitope_residues),
        "best_epoch": best_epoch,
        "early_stopped": early_stopped_flag,
    },
    "validation_losses": {
        "total_loss": float(val_stats["total_loss"]),
        "cls_loss": float(val_stats["cls_loss"]),
        "epi_loss": float(val_stats["epi_loss"]),
        "weighted_cls": float(val_stats["weighted_cls"]),
        "weighted_epi": float(val_stats["weighted_epi"]),
    },
    "validation_classification_metrics": val_classification_metrics,
    "validation_residue_metrics": val_residue_metrics,
    "test_metrics": test_metrics,
    "test_residue_metrics": test_residue_metrics,
}

with METRICS_PATH.open("w") as handle:
    json.dump(metrics_payload, handle, indent=2)

print("Validation classification metrics:")
print(json.dumps(val_classification_metrics, indent=2))
print("Validation residue metrics:")
print(json.dumps(val_residue_metrics, indent=2))
print("Test classification metrics:")
print(json.dumps(test_metrics, indent=2))
print("Test residue metrics:")
print(json.dumps(test_residue_metrics, indent=2))
print(f"Saved metrics to: {METRICS_PATH}")


In [ ]:
def compute_probe_metrics(labels: np.ndarray, scores: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    seq_len = len(labels)
    positives = int(labels.sum())

    if seq_len == 0 or positives == 0 or positives == seq_len:
        return {"auroc": np.nan, "auprc": np.nan, "precision_at_k": np.nan}

    auroc = roc_auc_score(labels, scores)
    auprc = average_precision_score(labels, scores)

    k = max(positives, 1)
    top_k = np.argsort(scores)[-k:]
    precision_at_k = float(labels[top_k].mean())

    return {
        "auroc": float(auroc),
        "auprc": float(auprc),
        "precision_at_k": precision_at_k,
    }


rng = np.random.default_rng(RANDOM_STATE)
probe_rows = []

for _, row in tqdm(epitope_probe_df.iterrows(), total=len(epitope_probe_df), desc="Probing splitB"):
    sequence = row["sequence"]
    epitope_labels = row["epitope_label"]
    base = {
        "accession": row["accession"],
        "seq_len": int(row["seq_len"]),
        "epitope_density": float(row["epitope_density"]),
        "n_epitope_residues": int(row["n_epitope_residues"]),
    }

    residue_scores = compute_residue_probabilities(model, tokenizer, sequence, DEVICE)
    probe_rows.append({**base, "method": "residue_head", **compute_probe_metrics(epitope_labels, residue_scores)})

    attention_scores = compute_attention_weights(model, tokenizer, sequence, DEVICE)
    probe_rows.append({**base, "method": "attention_weights", **compute_probe_metrics(epitope_labels, attention_scores)})

    ig_scores = compute_integrated_gradients(
        model,
        tokenizer,
        sequence,
        DEVICE,
        steps=IG_STEPS,
        normalize=False,
    )
    probe_rows.append({**base, "method": "integrated_gradients", **compute_probe_metrics(epitope_labels, ig_scores)})

    random_metrics = [
        compute_probe_metrics(epitope_labels, rng.uniform(0.0, 1.0, size=len(epitope_labels)))
        for _ in range(N_RANDOM_DRAWS)
    ]
    probe_rows.append({**base, "method": "random_mean", **mean_metric_dicts(random_metrics)})

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(PROBE_ROWS_PATH, index=False)
print(f"Saved row-wise probe metrics to: {PROBE_ROWS_PATH}")
probe_df.head()


In [ ]:
summary_rows = []
for method in ["residue_head", "attention_weights", "integrated_gradients", "random_mean"]:
    method_df = probe_df[probe_df["method"] == method]
    summary_rows.append(
        {
            "method": method,
            "auroc_mean": round(float(method_df["auroc"].dropna().mean()), 4),
            "auroc_sd": round(float(method_df["auroc"].dropna().std()), 4),
            "auprc_mean": round(float(method_df["auprc"].dropna().mean()), 4),
            "auprc_sd": round(float(method_df["auprc"].dropna().std()), 4),
            "precision_at_k_mean": round(float(method_df["precision_at_k"].dropna().mean()), 4),
            "precision_at_k_sd": round(float(method_df["precision_at_k"].dropna().std()), 4),
            "n_proteins": int(len(method_df)),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PROBE_SUMMARY_PATH, index=False)
print(summary_df.to_string(index=False))
print(f"Saved summary to: {PROBE_SUMMARY_PATH}")

comparison_df = None
if BASELINE_SUMMARY_CSV.exists():
    baseline_summary_df = pd.read_csv(BASELINE_SUMMARY_CSV)
    comparable_methods = ["attention_weights", "integrated_gradients", "random_mean"]
    comparison_df = baseline_summary_df[baseline_summary_df["method"].isin(comparable_methods)].merge(
        summary_df[summary_df["method"].isin(comparable_methods)],
        on="method",
        suffixes=("_baseline", "_mtl"),
    )
    for metric in ["auroc_mean", "auprc_mean", "precision_at_k_mean"]:
        comparison_df[f"delta_{metric}"] = (
            comparison_df[f"{metric}_mtl"] - comparison_df[f"{metric}_baseline"]
        ).round(4)
    comparison_df.to_csv(COMPARE_SUMMARY_PATH, index=False)
    print("\nBaseline vs MTL comparison")
    print(comparison_df.to_string(index=False))
    print(f"Saved comparison to: {COMPARE_SUMMARY_PATH}")
else:
    print(f"Baseline summary not found at: {BASELINE_SUMMARY_CSV}")


In [ ]:
plot_df = summary_df.melt(
    id_vars="method",
    value_vars=["auroc_mean", "auprc_mean", "precision_at_k_mean"],
    var_name="metric",
    value_name="value",
)

metric_labels = {
    "auroc_mean": "AUROC",
    "auprc_mean": "AUPRC",
    "precision_at_k_mean": "Precision@k",
}
plot_df["metric"] = plot_df["metric"].map(metric_labels)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="metric", y="value", hue="method")
plt.ylim(0, 1)
plt.title("Held-out Epitope Alignment on positives_splitB")
plt.tight_layout()
plt.savefig(PROBE_PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved plot to: {PROBE_PLOT_PATH}")


In [ ]:
if RUN_TARGET == "colab":
    import shutil as _shutil

    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    for _src, _dst_dir in [
        (CHECKPOINT_PATH, DRIVE_MODELS),
        (METRICS_PATH, DRIVE_RESULTS),
        (PROBE_ROWS_PATH, DRIVE_RESULTS),
        (PROBE_SUMMARY_PATH, DRIVE_RESULTS),
        (COMPARE_SUMMARY_PATH, DRIVE_RESULTS),
        (PROBE_PLOT_PATH, DRIVE_RESULTS),
    ]:
        if _src.exists():
            _shutil.copy2(_src, _dst_dir / _src.name)
            print(f"Copied to Drive: {_dst_dir / _src.name}")
        else:
            print(f"Skipped missing output: {_src}")
else:
    print("Local run: outputs saved to:")
    for _out_path in [
        CHECKPOINT_PATH,
        METRICS_PATH,
        PROBE_ROWS_PATH,
        PROBE_SUMMARY_PATH,
        COMPARE_SUMMARY_PATH,
        PROBE_PLOT_PATH,
    ]:
        print(f"  {_out_path}")


## Interpretation Guide

What to look for after running the notebook:
- `results/mtl_baseline_metrics.json`: sequence-level and flattened residue-level metrics for the checkpoint initialized from notebook 03.
- `train_cls_loss` vs `train_epi_loss` and their weighted counterparts: confirms whether protein classification or residue supervision is dominating optimization.
- `validation_residue_metrics` and `test_residue_metrics`: flattened residue AUROC/AUPRC over valid positive residues only.
- `results/mtl_probing_summary.csv`: held-out per-protein alignment scores on `positives_splitB.csv` for the residue head, attention weights, and integrated gradients.
- `results/mtl_vs_baseline_summary.csv`: change in attention- and IG-based alignment relative to notebook 03.
- If the residue head improves while protein metrics stay stable, the auxiliary task is adding localized supervision without sacrificing allergen classification.
